# 03 - Engenharia de features e modelagem

## Baseline de referência (v2)
Modelo `HistGradientBoostingClassifier` com 17 features disponíveis na submissão
(`tipo_contrato`, `valor_credito`, `valor_bem`, `valor_parcela` e o cadastral, sem `sexo`),
split OOT 80/20 com embargo de 360 dias e teste de 20.822 contratos.

| Métrica (teste completo, com embargo) | v1 (36 features) | v2 (17 features) |
|---|---|---|
| AUC | 0,584 | 0,556 |
| KS | 0,134 | 0,093 |

Walk-forward v2 (folds 2 a 4): AUC 0,575 / 0,612 / 0,582.

**Por que é a referência:** o v1 usava colunas que não existem na submissão
(`tipo_cliente`, `indicador_seguro_contratado`, condições do contrato), então não é
reproduzível em produção. O v2 mede o desempenho real do que estará disponível na
submissão. Toda feature nova será comparada ao v2, no mesmo teste e com os mesmos parâmetros.

In [10]:
import importlib
import sys
from pathlib import Path
sys.path.append("..")

import src.functions as fn #Funcoes criadas no projeto
importlib.reload(fn)

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.inspection import permutation_importance

In [11]:
COL_DATA, COL_CLIENTE, COL_TARGET = "data_decisao", "id_cliente", "OVER60MOB12"
PROP_TREINO, EMBARGO_DIAS, N_FOLDS = 0.80, 360, 4
MIN_MAUS_TREINO_FOLD, MIN_MAUS_VAL_FOLD = 300, 100

# Valores esperados (validação de reprodutibilidade)
N_TREINO_ESPERADO, N_SEM_ROTULO_ESPERADO = 104157, 107787

DIR_PROCESSADOS = Path("../dados/processados")
ARQUIVOS = {
    "abt_completa": "abt_completa.parquet",
    "abt_submissao": "abt_submissao.parquet",
}

for arquivo in ARQUIVOS.values():
    assert (DIR_PROCESSADOS / arquivo).exists(), f"Arquivo não encontrado: {DIR_PROCESSADOS / arquivo}"

base_completa = pd.read_parquet(DIR_PROCESSADOS / ARQUIVOS["abt_completa"])
base_submissao = pd.read_parquet(DIR_PROCESSADOS / ARQUIVOS["abt_submissao"])

# Referência temporal única: data_decisao no treino, data_solicitacao na submissão
base_completa["data_ref"] = base_completa[COL_DATA]
base_submissao["data_ref"] = base_submissao["data_solicitacao"]

# Treino: validação estrita
assert base_completa["data_ref"].notna().all(), "data_ref nula em base_completa"

# Submissão: diagnóstico (não interrompe a execução)
n_nulos_sub = int(base_submissao["data_ref"].isna().sum())
print(f"base_submissao: data_ref nula em {n_nulos_sub} de {len(base_submissao)} "
      f"({n_nulos_sub / len(base_submissao):.4%})")
print("dtype data_solicitacao:", base_submissao["data_solicitacao"].dtype)
colunas_data_flag = [c for c in base_submissao.columns if "data" in c or "flag" in c]
print("colunas de data/flag:", colunas_data_flag)
print(base_submissao.loc[base_submissao["data_ref"].isna(), colunas_data_flag].head(5))

# Separação: rotulados (treino) e sem rótulo (somente Reject Inference no treino)
mask_rotulado = base_completa[COL_TARGET].notna()
base_treino = base_completa.loc[mask_rotulado].copy()
base_sem_rotulo = base_completa.loc[~mask_rotulado].copy()

assert len(base_treino) == N_TREINO_ESPERADO, "base_treino com tamanho inesperado"
assert len(base_sem_rotulo) == N_SEM_ROTULO_ESPERADO, "base_sem_rotulo com tamanho inesperado"

print("base_completa:", base_completa.shape)
print("base_treino:", base_treino.shape, "| base_sem_rotulo:", base_sem_rotulo.shape)
print("base_submissao:", base_submissao.shape, "| data_ref nula:", int(base_submissao["data_ref"].isna().sum()))

base_submissao: data_ref nula em 51 de 50400 (0.1012%)
dtype data_solicitacao: datetime64[us]
colunas de data/flag: ['data_solicitacao', 'data_invalida', 'data_nascimento', 'data_nascimento_invalida', 'data_ref']
     data_solicitacao data_invalida data_nascimento  data_nascimento_invalida  \
1713              NaT          True      2001-03-04                     False   
3132              NaT          True      1974-12-21                     False   
4396              NaT          True      1984-11-23                     False   
6004              NaT          True      1989-06-22                     False   
6605              NaT          True      1985-02-03                     False   

     data_ref  
1713      NaT  
3132      NaT  
4396      NaT  
6004      NaT  
6605      NaT  
base_completa: (211944, 52)
base_treino: (104157, 52) | base_sem_rotulo: (107787, 52)
base_submissao: (50400, 23) | data_ref nula: 51


In [12]:
# ============================================================
# Split OOT e baseline v2 (17 features disponíveis na submissão)
# ============================================================
# Features comuns entre treino e submissão, sem chaves, target, datas e apoio
APOIO_V2 = {
    COL_CLIENTE, "id_contrato", COL_TARGET, COL_DATA, "data_ref",
    "data_solicitacao", "data_nascimento", "data_invalida",
    "data_nascimento_invalida",
}
FEATURES_V2 = [
    c for c in base_treino.columns
    if c in base_submissao.columns and c not in APOIO_V2
]
print(f"FEATURES_V2: {len(FEATURES_V2)} colunas")
print(FEATURES_V2)
assert len(FEATURES_V2) == 17, "FEATURES_V2 deve ter 17 colunas"

# Split temporal: com e sem embargo, teste idêntico nos dois cenários
tr_sem, teste, data_corte = fn.dividir_oot(base_treino, COL_DATA, PROP_TREINO, 0)
tr_com, _, _ = fn.dividir_oot(base_treino, COL_DATA, PROP_TREINO, EMBARGO_DIAS)
print("data_corte:", data_corte)
print("tr_sem:", len(tr_sem), "| tr_com:", len(tr_com), "| teste:", len(teste))

# Baseline v2: deve reproduzir AUC 0,556 e KS 0,093 (com embargo, completo)
res_v2 = fn.avaliar_oot(
    {"sem_embargo": tr_sem, "com_embargo": tr_com},
    teste, FEATURES_V2, COL_TARGET, COL_CLIENTE,
)
display(res_v2)

wf_v2 = fn.walk_forward(
    tr_com, FEATURES_V2, COL_DATA, COL_CLIENTE, COL_TARGET,
    N_FOLDS, EMBARGO_DIAS, MIN_MAUS_TREINO_FOLD, MIN_MAUS_VAL_FOLD,
)
display(wf_v2)

FEATURES_V2: 17 colunas
['tipo_contrato', 'valor_credito', 'valor_bem', 'valor_parcela', 'qtd_filhos', 'qtd_membros_familia', 'renda_anual', 'tipo_renda', 'ocupacao', 'tipo_organizacao', 'nivel_educacao', 'estado_civil', 'tipo_moradia', 'possui_carro', 'possui_imovel', 'nota_regiao_cliente', 'nota_regiao_cliente_cidade']
data_corte: 2023-06-06 00:00:00
tr_sem: 83335 | tr_com: 59535 | teste: 20822


,cenario,recorte,n,taxa_maus,auc,ks
0,sem_embargo,completo,20822,0.078667,0.567505,0.112604
1,sem_embargo,novos,6041,0.129780,0.536129,0.070117
2,sem_embargo,recorrentes,14781,0.057777,0.572694,0.132593
3,sem_embargo,treino (in-sample),83335,0.069695,0.705587,0.299345
4,com_embargo,completo,20822,0.078667,0.555977,0.092856
5,com_embargo,novos,8765,0.112151,0.541274,0.066538
6,com_embargo,recorrentes,12057,0.054325,0.574411,0.149623
7,com_embargo,treino (in-sample),59535,0.071790,0.712512,0.304535


,fold,val_inicio,val_fim,n_treino,maus_treino,n_val,maus_val,auc,ks,features_removidas,status
0,1,2018-07-02,2019-08-29,1842,116,11894,957,NaN,NaN,,ignorado: maus insuficientes
1,2,2019-08-29,2020-10-17,9740,684,11911,965,0.574617,0.135236,,ok
2,3,2020-10-17,2021-09-19,18058,1454,11903,841,0.612181,0.179185,,ok
3,4,2021-09-19,2022-06-11,25574,2186,11904,808,0.581797,0.131333,,ok


In [13]:
# ============================================================
# Grupo A: features sem histórico (idade e razões)
# ============================================================
importlib.reload(fn)

FEATURES_A_NOVAS = [
    "idade_anos", "razao_credito_parcela", "razao_credito_bem",
    "razao_credito_renda", "razao_parcela_renda",
]

base_treino = fn.criar_features_grupo_a(base_treino)
base_submissao = fn.criar_features_grupo_a(base_submissao)

# Validações: sem infinitos e nulos esperados
for nome, df in {"treino": base_treino, "submissao": base_submissao}.items():
    print(f"\n{nome}")
    print(df[FEATURES_A_NOVAS].describe(percentiles=[.01, .5, .99]).T[["min", "1%", "50%", "99%", "max"]])
    print("nulos:", df[FEATURES_A_NOVAS].isna().sum().to_dict())
    assert not np.isinf(df[FEATURES_A_NOVAS].to_numpy(dtype="float64")).any(), "infinito em features"


treino
                             min         1%        50%        99%         max
idade_anos            -12.807666  19.024559  40.232717  64.274552   67.920602
razao_credito_parcela   0.000000   0.000000   9.314494  35.977338  200.000000
razao_credito_bem       0.710710   0.805556   1.000000   1.206250    1.392618
razao_credito_renda     0.000000   0.000000   0.580000   8.666667   38.139083
razao_parcela_renda     0.000000   0.001161   0.064542   0.525442    5.564369
nulos: {'idade_anos': 184, 'razao_credito_parcela': 219, 'razao_credito_bem': 7072, 'razao_credito_renda': 526, 'razao_parcela_renda': 527}

submissao
                            min         1%        50%        99%          max
idade_anos            -4.906229  21.697467  43.268994  67.258700    68.993840
razao_credito_parcela  4.068335   6.189756  20.000000  43.780493  3792.667509
razao_credito_bem      0.300000   0.819481   1.105603   1.475200   152.800000
razao_credito_renda    0.001437   0.341918   2.831360  12.798

In [14]:
# ============================================================
# Diagnóstico do Grupo A (sem métricas de modelo)
# ============================================================
IDADE_MIN, IDADE_MAX = 18, 100
LIM_RAZAO_EXTREMA = {"razao_credito_parcela": 100, "razao_credito_bem": 2, "razao_credito_renda": 50}

for nome, df in {"treino": base_treino, "submissao": base_submissao}.items():
    idade = df["idade_anos"]
    fora_idade = idade.notna() & ((idade < IDADE_MIN) | (idade > IDADE_MAX))
    print(f"\n{nome}")
    print("idade fora de [18, 100]:", int(fora_idade.sum()))
    print("idade negativa:", int((idade < 0).sum()))
    print("valor_credito <= 0:", int((df["valor_credito"] <= 0).sum()))
    print("valor_bem <= 0:", int((df["valor_bem"] <= 0).sum()))
    print("valor_parcela <= 0:", int((df["valor_parcela"] <= 0).sum()))
    for col, lim in LIM_RAZAO_EXTREMA.items():
        print(f"{col} > {lim}:", int((df[col] > lim).sum()))

# Amostra de linhas extremas na submissão
extremos = base_submissao["razao_credito_bem"] > LIM_RAZAO_EXTREMA["razao_credito_bem"]
print(base_submissao.loc[extremos, ["valor_credito", "valor_bem", "valor_parcela", "renda_anual", "data_ref", "data_nascimento"]].head(10))


treino
idade fora de [18, 100]: 653
idade negativa: 76
valor_credito <= 0: 3729
valor_bem <= 0: 87
valor_parcela <= 0: 218
razao_credito_parcela > 100: 3
razao_credito_bem > 2: 0
razao_credito_renda > 50: 0

submissao
idade fora de [18, 100]: 75
idade negativa: 37
valor_credito <= 0: 0
valor_bem <= 0: 0
valor_parcela <= 0: 67
razao_credito_parcela > 100: 67
razao_credito_bem > 2: 132
razao_credito_renda > 50: 66
      valor_credito  valor_bem  valor_parcela  renda_anual   data_ref  \
607      27000000.0   270000.0        13500.0     180000.0 2025-02-23   
1293       942300.0   329805.0        34749.0     189000.0 2025-02-06   
1331       270000.0    94500.0        13500.0     180000.0 2025-02-20   
1360       180000.0    63000.0         9000.0      90000.0 2025-02-13   
1378     45000000.0   450000.0        22018.5     225000.0 2025-02-16   
1412     49049550.0   454500.0        29335.5     112500.0 2025-02-18   
1997       630000.0   220500.0        34308.0     238500.0 2025-02-17   

In [15]:
# ============================================================
# Diagnóstico para definição de políticas e tratamento
# ============================================================
# 1) Distribuição da razão crédito/bem na submissão: separar grupo 1 e grupo 2
r = base_submissao["razao_credito_bem"]
faixas_razao = pd.cut(r, bins=[0, 1.5, 2, 3, 10, 50, np.inf], right=False)
print(faixas_razao.value_counts(sort=False, dropna=False))

# Grupo 2: razão ~ 2,857 (bem = 35% do crédito)
grupo2 = r.between(2.8, 2.9)
print("razão ~2,857:", int(grupo2.sum()))

# Grupo 1: razão do crédito/100 vs bem, deve voltar para faixa plausível
grupo1 = r > 10
razao_corrigida = (base_submissao["valor_credito"] / 100) / base_submissao["valor_bem"]
print("grupo 1:", int(grupo1.sum()))
print(razao_corrigida[grupo1].describe(percentiles=[.05, .5, .95]))

# 2) Treino: crédito <= 0
cred0 = base_treino["valor_credito"] <= 0
print("\ncrédito <= 0 no treino:", int(cred0.sum()))
print("taxa de maus (crédito <= 0):", round(base_treino.loc[cred0, COL_TARGET].mean(), 4))
print("taxa de maus (crédito > 0): ", round(base_treino.loc[~cred0, COL_TARGET].mean(), 4))
print(base_treino.loc[cred0].filter(regex="status|tipo_contrato|data_decisao").head())

# 3) Impacto de cada política candidata na submissão
politicas = {
    "razao_credito_bem > 10": r > 10,
    "razao_credito_bem > 2 (inclui grupo 2)": r > 2,
    "valor_parcela <= 0": base_submissao["valor_parcela"] <= 0,
    "renda_anual <= 0 ou nula": ~(base_submissao["renda_anual"] > 0),
}
for nome, m in politicas.items():
    print(f"{nome}: {int(m.sum())} ({m.mean():.3%})")

razao_credito_bem
[0.0, 1.5)      47513
[1.5, 2.0)        291
[2.0, 3.0)         73
[3.0, 10.0)         0
[10.0, 50.0)        0
[50.0, inf)        63
NaN              2460
Name: count, dtype: int64
razão ~2,857: 67
grupo 1: 63
count    63.000000
mean      1.109808
std       0.127682
min       0.844915
5%        1.000000
50%       1.080386
95%       1.316812
max       1.528000
dtype: float64

crédito <= 0 no treino: 3729
taxa de maus (crédito <= 0): 0.0461
taxa de maus (crédito > 0):  0.0724
                tipo_contrato status_contrato data_decisao
10     emprestimo em dinheiro        aprovado   2022-09-15
192  financiamento de consumo        aprovado   2021-10-23
221  financiamento de consumo        aprovado   2021-11-25
232    emprestimo em dinheiro        aprovado   2022-12-26
239    emprestimo em dinheiro        aprovado   2018-10-23
razao_credito_bem > 10: 63 (0.125%)
razao_credito_bem > 2 (inclui grupo 2): 132 (0.262%)
valor_parcela <= 0: 67 (0.133%)
renda_anual <= 0 ou nula: 232